In [1]:
import pandas as pd 
import numpy as np 
import os 
import sys 
import re 
import json 
from statsmodels.stats.inter_rater import fleiss_kappa


In [2]:
path_to_jllm = "../../../data/conv-trs/ecir-2026/direct-reasoner/cleaned"
path_to_human = "../../../data/conv-trs/ecir-2026/human-eval"

jllm_name = "evals_cleaned_v1.csv"

In [3]:
deepseek = pd.read_csv(f"{path_to_jllm}/deepseek_{jllm_name}")
gpt = pd.read_csv(f"{path_to_jllm}/gpt5_{jllm_name}")
gemini = pd.read_csv(f"{path_to_jllm}/gemini2.5pro_{jllm_name}")

jllms = {
    "deepseek": deepseek, 
    "gpt": gpt, 
    "gemini": gemini
}

In [6]:
deepseek.head()

,query_id,query,judge_model,experiment,rec_llm_L1,rec_llm_L2,best_list,overall_explanation,overall_confidence,relevance_comparison,...,sustainability_comparison,sustainability_explanation,sustainability_confidence,popularity_balance_comparison,popularity_balance_explanation,popularity_balance_confidence,relevance_score,diversity_score,sustainability_score,popularity_balance_score
0,c_p_0_pop_low_easy,Cheap European city break in February.,deepseek-ai/deepseek-v3.1-maas,run_common_human_queries_deepseek,gemini_2_5_flash,gpt_4o,L1,L1 demonstrates superior relevance by specific...,0.85,Much more L1 than L2,...,Slightly more L1,"L1 explicitly mentions fewer crowds, walkabili...",0.80,About the same,Both lists mix well-known destinations (Krakow...,0.8,2,0,1,0
1,c_p_0_pop_low_medium,"Suggest some less-known, budget-friendly Europ...",deepseek-ai/deepseek-v3.1-maas,run_common_human_queries_deepseek,gemini_2_5_flash,gpt_4o,L1,L1 demonstrates superior alignment with the qu...,0.90,Much more L1 than L2,...,Slightly more L1,"L1 explicitly mentions walkability, car-free c...",0.80,Much more L1 than L2,L1 focuses exclusively on lesser-known Eastern...,0.9,2,0,1,2
2,c_p_0_pop_low_hard,Trip to a less-visited European city with hist...,deepseek-ai/deepseek-v3.1-maas,run_common_human_queries_deepseek,gemini_2_5_flash,gpt_4o,L1,L1 demonstrates slightly better alignment with...,0.75,Slightly more L1,...,About the same,Both lists focus on less crowded alternatives ...,0.70,Slightly more L1,L1 includes cities like Sarajevo and Dresden t...,0.7,1,0,0,1
3,c_p_0_pop_low_sustainable,Low budget European city break in April with g...,deepseek-ai/deepseek-v3.1-maas,run_common_human_queries_deepseek,gemini_2_5_flash,gpt_4o,L1,L1 offers slightly better geographic diversity...,0.80,About the same,...,About the same,"Both lists feature compact, walkable cities wi...",0.70,Slightly more L1,L1 includes Lisbon (more famous) alongside les...,0.8,0,1,0,1
4,c_p_0_pop_medium_easy,European city break in January with medium cro...,deepseek-ai/deepseek-v3.1-maas,run_common_human_queries_deepseek,gemini_2_5_flash,gpt_4o,L2,L2 provides a slightly better overall balance ...,0.80,About the same,...,Slightly more L1,L1 explicitly mentions sustainable transportat...,0.85,Slightly more L2,L2 includes less mainstream destinations like ...,0.8,0,-1,1,-1


In [47]:
human = pd.read_csv(f"{path_to_human}/survey_parsed.csv")
human = human[human['annotator'] != 'Comet [AI]']

In [48]:
filtered_human = human[human.groupby('query_id')['query_id'].transform('count') >= 3]
filtered_human.dropna(inplace=True)

/var/folders/07/81pmlyln5szbq49h1zdqjq1h0000gn/T/ipykernel_44850/1125450700.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_human.dropna(inplace=True)


In [49]:
jllm_cols = ['relevance_score','diversity_score','sustainability_score','popularity_balance_score']
human_cols = ['relevance', 'diversity', 'sustainability', 'popularity']

In [50]:
# get dataframes 
def get_df(eval_df, metric, jllm=False):
    if "popularity" in metric: 
        if jllm: 
            metric = "popularity_balance"

    if jllm:         
        df = eval_df[['query_id', f"{metric}_score"]]
        return df
    
    else: 
        df = eval_df[['query_id', 'annotator', f"{metric}"]]
        pivoted = df.pivot_table(
            index=['query_id'],
            columns='annotator',
            values=metric
        ).reset_index()

        return pivoted

def merge_df(human_df, metric):
    df = get_df(human_df, metric)
    # df.dropna(axis=1, inplace=True)

    for name, jllm in jllms.items():
        
        subset = get_df(jllm, metric, jllm=True) 
        df = pd.merge(
            left=df, 
            right=subset, 
            on='query_id'
        )
        if "popularity" in metric: 
            df.rename(columns={
                f'popularity_balance_score': f'{name}'
            }, inplace=True)
        else:
            df.rename(columns={
                f'{metric}': f'{name}'
            }, inplace=True)
    
    return df

In [51]:
metrics = {
    'relevance': pd.DataFrame(), 
    'diversity': pd.DataFrame(), 
    'popularity': pd.DataFrame(), 
    'sustainability': pd.DataFrame()
}

In [52]:
for metric in metrics: 
    metrics[metric] = merge_df(human, metric=metric)

In [53]:
metrics['popularity']

,query_id,Dana Marti,Ewald,Tejas Srinivasan,Wolfgang,Yas,deepseek,gpt,gemini
0,c_p_0_pop_high_easy,1.0,-1.0,NaN,-1.0,0.0,-2,-2,-2
1,c_p_0_pop_high_hard,-1.0,-1.0,NaN,0.0,NaN,-1,-1,-1
2,c_p_0_pop_high_medium,-1.0,1.0,NaN,0.0,NaN,-1,-2,-1
3,c_p_0_pop_low_easy,0.0,-1.0,NaN,0.0,-1.0,0,-1,0
4,c_p_0_pop_low_hard,0.0,-1.0,NaN,-3.0,-3.0,1,0,-1
...,...,...,...,...,...,...,...,...,...
96,c_p_8_pop_medium_sustainable,1.0,-1.0,NaN,0.0,1.0,1,1,1
97,c_p_97_pop_low_sustainable,0.0,0.0,NaN,0.0,NaN,1,-1,1
98,c_p_9_pop_high_hard,1.0,-1.0,-1.0,-1.0,NaN,-1,-1,-2
99,c_p_9_pop_low_easy,1.0,2.0,NaN,1.0,NaN,2,-1,-1


### Agreement using Fleiss Kappa

In [54]:
import numpy as np

def compute_fleiss_kappa(df, metric):
    # Initialize count matrix: rows=items, columns=categories
    df = df.dropna()

    # Define all possible rating categories
    categories = [-3, -2, -1, 0, 1, 2]
    n_annotators = df.shape[1]

    # Initialize count matrix: rows=items, columns=categories
    ratings_matrix = np.zeros((len(df), len(categories)), dtype=int)

    # Count how many annotators gave each category per item
    for i, cat in enumerate(categories):
        ratings_matrix[:, i] = (df == cat).sum(axis=1)

    # Check that every row sums to the same number of annotators
    row_sums = ratings_matrix.sum(axis=1)
    if not np.all(row_sums == n_annotators):
        raise ValueError(
            f"Each item must have {n_annotators} ratings — found varying counts: {row_sums}"
        )

    # Compute Fleiss' Kappa
    kappa = fleiss_kappa(ratings_matrix)
    print(f"Fleiss’ Kappa for {metric}: {kappa:.3f}")

    return kappa



In [55]:
for metric, df in metrics.items(): 
    new_df = df.drop(columns=['query_id'])
    compute_fleiss_kappa(new_df, metric)


Fleiss’ Kappa for relevance: -0.101
Fleiss’ Kappa for diversity: 0.135
Fleiss’ Kappa for popularity: -0.051
Fleiss’ Kappa for sustainability: 0.129
